# 08 — Khôi phục phiên 2026-10-15 (v6p5 plateau, audit, structural consistency) trên máy mới
Chạy lần lượt từ trên xuống. Mỗi cell bỏ qua nếu việc đã xong (chạy lại an toàn).

1. CONFIG: Camber key + CLI
2. CODE: clone repo / áp `repo.bundle` từ Camber nếu GitHub chưa có commit của phiên
3. ENV: cài package vào kernel hiện tại
4. DATASET v6 (6.9 GB, tải từ Camber) + dựng v7 (chỉ nhãn, ảnh hard-link từ v6)
5. SESSION: weights, cache dự đoán, kết quả phụ
6. VERIFY: tái tạo số chính từ cache (không cần GPU)
7. RQ: chấm bảng xác minh `verify_blind.csv` khi đã điền

Bối cảnh đầy đủ: `docs/SESSION_2026-10-15_v6p5_plateau.md`. Cần: `CAMBER_API_KEY` (trong `.env` của repo, biến môi trường,
hoặc nhập khi được hỏi). Nếu mở notebook này ở chỗ chưa có repo, đặt nó vào thư mục cha mong muốn rồi chạy; cell 2 sẽ clone.

In [ ]:
# 1. CONFIG
import os, sys, json, shutil, subprocess, tarfile, hashlib, getpass, platform
from pathlib import Path

GIT_URL = "https://github.com/Eau-Claire/Uav-Powerline-Defect-Detection.git"
BRANCH = "v6p5-plateau-yolo26n"
SESSION_COMMIT = "6c1bd54dfbda2b01c3872202831baaf743b3f321"         # commit chứa script + notebook này
STASH = "stash://bunpmc/projects/yolov11sdi"
SESSION = f"{STASH}/checkpoints/v6p5_v7_session_20261015"
NEED_DATASET = True        # False nếu chỉ cần xem kết quả / weights (không train, không eval lại)
INSTALL = True             # cài package vào kernel đang chạy (sys.executable)

here = Path.cwd()
REPO = next((p for p in [here, *here.parents] if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir()),
            here / "Uav-Powerline-Defect-Detection")
for f in [REPO / ".env", here / ".env"]:
    if f.is_file():
        for line in f.read_text().splitlines():
            if "=" in line and not line.lstrip().startswith("#"):
                k, v = line.split("=", 1); os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
if not os.environ.get("CAMBER_API_KEY"):
    os.environ["CAMBER_API_KEY"] = getpass.getpass("CAMBER_API_KEY: ")
CAMBER = shutil.which("camber") or str(Path.home() / ".camber/bin/camber")
if not Path(CAMBER).exists() and platform.system() != "Windows":
    subprocess.run("curl -sL https://cli.cambercloud.com/install-v2.sh | bash", shell=True, check=True)
assert Path(CAMBER).exists() or shutil.which(CAMBER), "Không thấy Camber CLI (Windows: cài tay, đặt CAMBER_BIN)"

def camber_cp(src, dst):
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([CAMBER, "stash", "cp", src, str(dst)], check=True, stdout=subprocess.DEVNULL)
print("REPO:", REPO, "| camber:", CAMBER)

In [ ]:
# 2. CODE: repo ở đúng branch, có commit của phiên (GitHub có thể chưa có -> lấy từ repo.bundle trên Camber)
run = lambda *a: subprocess.run(["git", "-C", str(REPO), *a], text=True, capture_output=True)
has = lambda c: run("cat-file", "-e", f"{c}^{{commit}}").returncode == 0
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "-b", BRANCH, GIT_URL, str(REPO)], check=True)
target = SESSION_COMMIT
if not has(target): run("fetch", "origin", BRANCH)
if not has(target):
    bundle = REPO.parent / "v6p5_session_20261015.bundle"
    if not bundle.is_file(): camber_cp(f"{SESSION}/repo.bundle", bundle)
    r = run("fetch", str(bundle), f"{BRANCH}:refs/remotes/bundle/{BRANCH}"); assert r.returncode == 0, r.stderr
    target = f"refs/remotes/bundle/{BRANCH}"; print("lấy commit từ bundle Camber")
if run("rev-parse", "--abbrev-ref", "HEAD").stdout.strip() != BRANCH:
    r = run("checkout", BRANCH); assert r.returncode == 0, r.stderr
if run("merge-base", "--is-ancestor", target, "HEAD").returncode != 0:
    if run("status", "--porcelain", "--untracked-files=no").stdout.strip():
        raise RuntimeError("Repo có thay đổi chưa commit; commit/stash trước rồi chạy lại cell này")
    r = run("merge", "--ff-only", target); assert r.returncode == 0, r.stderr
print("HEAD:", run("log", "-1", "--oneline").stdout.strip())
print("Nếu GitHub chưa có commit này, từ máy có quyền push: git push origin", BRANCH)

In [ ]:
# 3. ENV: package của repo + phụ thuộc của các script phiên này (torch CUDA lấy theo pip mặc định)
if INSTALL:
    pip = [sys.executable, "-m", "pip", "install", "-q"]
    subprocess.run(pip + ["-e", f"{REPO}[stage,train,notebook,dev]", "-r", str(REPO / "requirements.txt")], check=True)
    subprocess.run(pip + ["scikit-learn", "ensemble-boxes", "tabulate", "opencv-python-headless"], check=True)
    # tuỳ chọn, chỉ cho prototype diffusion: diffusers transformers accelerate
import torch, ultralytics
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| ultralytics", ultralytics.__version__)
assert ultralytics.__version__ == "8.4.172", "Cần ultralytics==8.4.172 (distill_model, cùng số liệu)"

In [ ]:
# 4. DATASET v6 (Camber) -> datasets/v6p5_v6 ; v7 = nhãn v7 + ảnh hard-link từ v6
import yaml
DSD = REPO / "datasets"; DS6, DS7 = DSD / "v6p5_v6", DSD / "v6p5_v7"
if NEED_DATASET and not (DS6 / "manifest.json").is_file():
    for f in ("v6p5_v6.tar.sha256", "v6p5_v6.package.json", "v6p5_v6.tar"):
        if not (DSD / f).is_file(): print("tải", f); camber_cp(f"{STASH}/datasets/v6p5_v6/{f}", DSD / f)
    h = hashlib.sha256()
    with open(DSD / "v6p5_v6.tar", "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""): h.update(chunk)
    assert h.hexdigest() == (DSD / "v6p5_v6.tar.sha256").read_text().split()[0], "sha256 sai -> xoá tar, chạy lại"
    tarfile.open(DSD / "v6p5_v6.tar").extractall(DSD); (DSD / "v6p5_v6.tar").unlink()
if DS6.is_dir():
    for y in ["data.yaml", "data_val_clean.yaml", "data_val_clean_v6.yaml"]:
        if (DS6 / y).is_file():
            cfg = yaml.safe_load((DS6 / y).read_text()); cfg["path"] = str(DS6); (DS6 / y).write_text(yaml.safe_dump(cfg, sort_keys=False))
    print("v6:", {s: len(list((DS6 / "images" / s).glob("*.jpg"))) for s in ["train", "val", "val_clean", "val_clean_v6"]})

In [ ]:
# 5. SESSION: weights (.pt), cache dự đoán (sys_*.json, pred_*.json), nhãn v7, file phụ -> đúng đường dẫn trong repo
TARS = ["weights.tar", "preds.tar", "dataset_v7_labels.tar", "misc.tar"]
dl = REPO.parent / "v6p5_session_20261015"
for t in TARS:
    if not (dl / t).is_file(): print("tải", t); camber_cp(f"{SESSION}/{t}", dl / t)
    with tarfile.open(dl / t) as tf: tf.extractall(REPO)       # tar lưu đường dẫn tương đối từ gốc repo
# cache dự đoán ghi đường dẫn weights tuyệt đối của máy cũ -> đổi sang máy này để script dùng lại cache, không chạy lại
OLD = "/root/projects/Uav-Powerline-Defect-Detection"
for f in list((REPO / "data/label_audit/preflight").glob("sys_*.json")) + list(REPO.glob("artifacts/**/pred_*.json")):
    s = f.read_text()
    if OLD in s[:2000]: f.write_text(s.replace(OLD, str(REPO)))
# v7: ảnh = ảnh v6 (hard link; copy nếu khác ổ)
if DS6.is_dir() and (DS7 / "labels/train").is_dir():
    for split in ["train", "val", "val_clean", "val_clean_v6"]:
        (DS7 / "images" / split).mkdir(parents=True, exist_ok=True)
        for im in (DS6 / "images" / split).iterdir():
            d = DS7 / "images" / split / im.name
            if not d.exists():
                try: os.link(im, d)
                except OSError: shutil.copy2(im, d)
    for y in ["data.yaml", "data_val_clean.yaml", "data_val_clean_v6.yaml"]:
        cfg = yaml.safe_load((DS7 / y).read_text()); cfg["path"] = str(DS7); (DS7 / y).write_text(yaml.safe_dump(cfg, sort_keys=False))
W = {"v5_orig (server hiện tại)": "artifacts/v6p5_v6_teachers/init/teacher_v5.pt",
     "soup_g_v6 (ngang v5_orig, có nhãn v6)": "artifacts/v6p5_v7_gentle/soups/soup_g_v6.pt",
     "student n_v5 (Pi hiện tại)": "artifacts/v6p5_v6_teachers/init/student_n_v5.pt",
     "student gn_v6": "artifacts/v6p5_v7_gentle/runs/gn_v6/weights/best.pt"}
for k, p in W.items(): print(f"{k:40s} {'OK' if (REPO / p).is_file() else 'THIẾU'}  {p}")

In [ ]:
# 6. VERIFY: tái tạo MEAN_5 hệ 2 tầng từ cache (CPU). Kỳ vọng: v5_orig 0.8283 / không rò 0.8158; soup_g_v6 0.8281 / 0.8165
import numpy as np
sys.path.insert(0, str(REPO / "data/label_audit/preflight"))
from system_two_tier import flagged
from system_two_tier_bootstrap import prep, wap
PF = REPO / "data/label_audit/preflight"
leak = {v for v, _ in json.loads((REPO / "artifacts/review_v7/val_train_near_dups.json").read_text())}
ALL = sorted((DS6 / "images/val_clean_v6").glob("*.jpg"))
def gts(imgs):
    G = {c: {} for c in range(1, 6)}
    for im in imgs:
        x = np.array([l.split() for l in (DS6 / "labels/val_clean_v6" / (im.stem + ".txt")).read_text().splitlines() if l.strip()], float).reshape(-1, 5)
        for c in range(1, 6): b = x[x[:, 0] == c][:, 1:]; G[c][im.name] = np.c_[b[:, :2] - b[:, 2:] / 2, b[:, :2] + b[:, 2:] / 2]
    return G
dets = lambda t: json.loads((PF / f"sys_{t}.json").read_text())["dets"]
flag = flagged(dets("nb22_pi_n_v5_640"), 0.05)
for name, imgs in [("toàn bộ", ALL), ("không rò", [p for p in ALL if p.name not in leak])]:
    idx = {p.name: i for i, p in enumerate(imgs)}; G = gts(imgs); out = []
    for k, t in [("v5_orig", "nb22_v5_orig"), ("soup_g_v6", "nb23_soup_g_v6"), ("g_v7", "nb23_g_v7")]:
        p6, p8 = dets(t + "_640"), dets(t + "_800")
        ap = [wap(*prep({n: (p8 if c == 4 else p6)[n] for n in idx}, G[c], c, flag, idx), np.ones(len(imgs))) for c in range(1, 6)]
        out.append(f"{k} {np.mean(ap):.4f}")
    print(f"{name:9s} ({len(imgs)} ảnh): " + " | ".join(out))

In [ ]:
# 7. RQ phần B: chấm bảng xác minh mù sau khi đã điền cột verdict trong artifacts/struct_audit/verify_blind.csv
#    verdict: correct | wrong_class:<cls> | bad_box | not_defect | duplicate | unsure
import pandas as pd
SA = REPO / "artifacts/struct_audit"
V = pd.read_csv(SA / "verify_blind.csv"); K = pd.read_csv(SA / "verify_key.csv")
done = V.verdict.notna() & (V.verdict.astype(str).str.strip() != "")
print(f"đã điền {done.sum()}/{len(V)}")
if done.sum():
    M = V[done].merge(K, on="vid"); M = M[M.verdict != "unsure"]
    M["error"] = M.verdict != "correct"
    rows = []
    for s in ["s_conf", "s_loss", "s_disagree", "s_struct"]:
        top = M.sort_values(s, ascending=False)
        for k in (10, 25, 50):
            t = top.head(k); rows.append({"method": s, "k": k, "n": len(t), "precision": t.error.mean()})
    print(pd.DataFrame(rows).pivot(index="method", columns="k", values="precision").round(2))
    conv = set(M.nlargest(50, "s_conf").vid) | set(M.nlargest(50, "s_loss").vid) | set(M.nlargest(50, "s_disagree").vid)
    only = M[M.vid.isin(set(M.nlargest(50, "s_struct").vid) - conv)]
    print(f"lỗi CHỈ cấu trúc tìm ra (top-50, không nằm trong top-50 nào khác): {int(only.error.sum())} / {len(only)}")
    print("baseline ngẫu nhiên (tỉ lệ lỗi chung trong mẫu):", round(M.error.mean(), 2))

## Tiếp theo (chi tiết trong `docs/SESSION_2026-10-15_v6p5_plateau.md`, mục 7)
1. Điền `verify_blind.csv` → chạy cell 7 → trả lời RQ phần B.
2. v8: bỏ 148 ảnh train trùng val; báo cáo MEAN_5 không rò + broken_strand tách drone / mặt đất.
3. Sửa val theo `artifacts/review_v7/README.md` (bird_nest thiếu, insulator_broken đáng ngờ, drone ≠ FO).
4. Thu ảnh drone thật có dây tưa; pre-label bằng teacher.
5. Train lại đầy đủ khi nhãn ổn (đối chứng v6 cùng công thức). Script tham khảo: `scripts/exp_v6p5_23_gentle_ft.py` (train/soup/eval dùng chung).